Import libraries

In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata
import random
import os
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Libraries loaded.")

Libraries loaded.


In [2]:
from google.colab import files

uploaded = files.upload()

Saving training_set_balanced.csv to training_set_balanced.csv


In [3]:
from google.colab import files

uploaded = files.upload()

Saving validation_set.csv to validation_set.csv


In [4]:
from google.colab import files

uploaded = files.upload()

Saving test_set.csv to test_set.csv


In [5]:
from google.colab import files

uploaded = files.upload()

Saving single_obfuscation_attacks.csv to single_obfuscation_attacks.csv


In [6]:
from google.colab import files

uploaded = files.upload()

Saving multi_obfuscation_attacks.csv to multi_obfuscation_attacks.csv


In [7]:
train_df = pd.read_csv("training_set_balanced.csv")
val_df = pd.read_csv("validation_set.csv")
test_df = pd.read_csv("test_set.csv")

single_df = pd.read_csv("single_obfuscation_attacks.csv")
multi_df = pd.read_csv("multi_obfuscation_attacks.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Clean test:", test_df.shape)
print("Single attacks:", single_df.shape)
print("Multi attacks:", multi_df.shape)

Train: (65307, 5)
Validation: (22771, 5)
Clean test: (22777, 5)
Single attacks: (42749, 10)
Multi attacks: (7872, 11)


In [8]:
# Identify columns
def find_column(df, candidates, required=True):
    for col in candidates:
        if col in df.columns:
            return col

    if required:
        raise ValueError(
            f"Could not find any of {candidates}. "
            f"Available columns: {list(df.columns)}"
        )

    return None

In [9]:
train_text_col = find_column(
    train_df,
    ["text", "comment_text", "tweet"]
)

val_text_col = find_column(
    val_df,
    ["text", "comment_text", "tweet"]
)

test_text_col = find_column(
    test_df,
    ["text", "comment_text", "tweet"]
)

label_col = find_column(
    train_df,
    ["abusive", "label", "class", "target"]
)

single_text_col = find_column(
    single_df,
    ["obfuscated_text", "text", "attacked_text"]
)

multi_text_col = find_column(
    multi_df,
    ["obfuscated_text", "text", "attacked_text"]
)

severity_single_col = find_column(
    single_df,
    ["severity", "severity_level"],
    required=False
)

severity_multi_col = find_column(
    multi_df,
    ["severity", "severity_level"],
    required=False
)

single_attack_col = find_column(
    single_df,
    [
        "technique",
        "obfuscation_type",
        "attack_type",
        "transformation",
        "method"
    ],
    required=False
)

multi_attack_col = find_column(
    multi_df,
    [
        "technique",
        "obfuscation_type",
        "attack_type",
        "combination",
        "techniques",
        "transformation"
    ],
    required=False
)

print("Training text:", train_text_col)
print("Label:", label_col)
print("Single attack text:", single_text_col)
print("Single attack type:", single_attack_col)
print("Multi attack text:", multi_text_col)
print("Multi attack type:", multi_attack_col)

Training text: text
Label: abusive
Single attack text: obfuscated_text
Single attack type: obfuscation_type
Multi attack text: obfuscated_text
Multi attack type: obfuscation_type


Prepare labels

In [10]:
label_map = {
    "OFF": 1,
    "NOT": 0,
    1: 1,
    0: 0,
    "1": 1,
    "0": 0
}

def convert_labels(series):
    mapped = series.map(label_map)

    if mapped.isna().any():
        print("Unknown labels:")
        print(series[mapped.isna()].unique())

    return mapped.astype(int)

In [11]:
y_train = convert_labels(train_df[label_col])
y_val = convert_labels(val_df[label_col])
y_test = convert_labels(test_df[label_col])

single_label_col = find_column(
    single_df,
    ["abusive", "label", "class", "target"]
)

multi_label_col = find_column(
    multi_df,
    ["abusive", "label", "class", "target"]
)

y_single = convert_labels(single_df[single_label_col])
y_multi = convert_labels(multi_df[multi_label_col])

Rule-based normalisation layer

In [12]:
CHAR_SUBSTITUTIONS = {
    "@": "a",
    "4": "a",
    "3": "e",
    "1": "i",
    "!": "i",
    "0": "o",
    "$": "s",
    "5": "s",
    "7": "t",
    "+": "t"
}

UNICODE_CONFUSABLES = {
    # Cyrillic
    "а": "a",
    "е": "e",
    "о": "o",
    "р": "p",
    "с": "c",
    "х": "x",
    "у": "y",
    "і": "i",

    # Greek / other common confusables
    "Α": "a",
    "Β": "b",
    "Ε": "e",
    "Ι": "i",
    "Κ": "k",
    "Μ": "m",
    "Ν": "n",
    "Ο": "o",
    "Ρ": "p",
    "Τ": "t",
    "Χ": "x"
}

In [13]:
def collapse_spaced_characters(text):
    pattern = r'(?<!\w)(?:[a-zA-Z]\s+){2,}[a-zA-Z](?!\w)'

    def merge(match):
        return re.sub(r"\s+", "", match.group(0))

    return re.sub(pattern, merge, text)

In [14]:
def normalise_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    # 1. Lowercase
    text = text.lower()

    # 2. Unicode normalisation
    text = unicodedata.normalize("NFKC", text)

    # 3. Convert Unicode confusables
    text = "".join(
        UNICODE_CONFUSABLES.get(char, char)
        for char in text
    )

    # 4. Reverse common character substitutions
    text = "".join(
        CHAR_SUBSTITUTIONS.get(char, char)
        for char in text
    )

    # 5. Collapse deliberately spaced characters
    text = collapse_spaced_characters(text)

    # 6. Remove symbols inserted BETWEEN letters
    # h.a.t.e -> hate
    text = re.sub(
        r'(?<=[a-z])[\.\-_*/\\|]+(?=[a-z])',
        '',
        text
    )

    # 7. Reduce excessive repeated punctuation
    text = re.sub(r'([!?.,])\1+', r'\1', text)

    # 8. Reduce extreme character repetition
    # idiiiiiot -> idiiot
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # 9. Normalise whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [15]:
# Apply normalisation
train_df["normalised_text"] = (
    train_df[train_text_col]
    .fillna("")
    .astype(str)
    .apply(normalise_text)
)

val_df["normalised_text"] = (
    val_df[val_text_col]
    .fillna("")
    .astype(str)
    .apply(normalise_text)
)

test_df["normalised_text"] = (
    test_df[test_text_col]
    .fillna("")
    .astype(str)
    .apply(normalise_text)
)

single_df["normalised_text"] = (
    single_df[single_text_col]
    .fillna("")
    .astype(str)
    .apply(normalise_text)
)

multi_df["normalised_text"] = (
    multi_df[multi_text_col]
    .fillna("")
    .astype(str)
    .apply(normalise_text)
)

Train LR + SVM

In [16]:
# n-gram
raw_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=2,
    sublinear_tf=True
)

X_train_raw = raw_vectorizer.fit_transform(
    train_df[train_text_col].fillna("").astype(str)
)

X_val_raw = raw_vectorizer.transform(
    val_df[val_text_col].fillna("").astype(str)
)

In [17]:
# LR
lr_raw = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=SEED
)

lr_raw.fit(X_train_raw, y_train)

lr_val_pred = lr_raw.predict(X_val_raw)

lr_val_f1 = f1_score(y_val, lr_val_pred)

print("LR validation F1:", lr_val_f1)

LR validation F1: 0.5650396729983169


In [18]:
# SVM
svm_raw = LinearSVC(
    class_weight="balanced",
    random_state=SEED
)

svm_raw.fit(X_train_raw, y_train)

svm_val_pred = svm_raw.predict(X_val_raw)

svm_val_f1 = f1_score(y_val, svm_val_pred)

print("SVM validation F1:", svm_val_f1)

SVM validation F1: 0.5730223123732252


In [19]:
# Validation performance
if svm_val_f1 >= lr_val_f1:
    best_ml_name = "SVM"
else:
    best_ml_name = "Logistic Regression"

print("Selected hybrid classifier:", best_ml_name)

Selected hybrid classifier: SVM


Fit a NEW vectorizer on normalised training data

In [20]:
hybrid_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=2,
    sublinear_tf=True
)

X_train_hybrid = hybrid_vectorizer.fit_transform(
    train_df["normalised_text"]
)

X_val_hybrid = hybrid_vectorizer.transform(
    val_df["normalised_text"]
)

Train classifier

In [21]:
if best_ml_name == "SVM":

    hybrid_model = LinearSVC(
        class_weight="balanced",
        random_state=SEED
    )

else:

    hybrid_model = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=SEED
    )

hybrid_model.fit(X_train_hybrid, y_train)

print("Hybrid model trained:", best_ml_name)

Hybrid model trained: SVM


In [22]:
# Check validation performance
hybrid_val_pred = hybrid_model.predict(X_val_hybrid)

print(
    classification_report(
        y_val,
        hybrid_val_pred,
        target_names=["NOT", "OFF"]
    )
)

              precision    recall  f1-score   support

         NOT       0.97      0.85      0.91     20051
         OFF       0.44      0.83      0.57      2720

    accuracy                           0.85     22771
   macro avg       0.70      0.84      0.74     22771
weighted avg       0.91      0.85      0.87     22771



Evaluation function

In [23]:
# Metrics
def calculate_metrics(y_true, y_pred):

    return {
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "accuracy": accuracy_score(
            y_true,
            y_pred
        )
    }

In [24]:
# Hybrid prediction function
def hybrid_predict(texts):

    normalised = (
        pd.Series(texts)
        .fillna("")
        .astype(str)
        .apply(normalise_text)
    )

    features = hybrid_vectorizer.transform(normalised)

    return hybrid_model.predict(features)

In [25]:
# Clean set evaluation
clean_predictions = hybrid_predict(
    test_df[test_text_col]
)

clean_hybrid_metrics = calculate_metrics(
    y_test,
    clean_predictions
)

clean_hybrid_metrics

{'precision': 0.4361094703681827,
 'recall': 0.8134410576569959,
 'f1': 0.5678031274032299,
 'accuracy': 0.851955920446064}

Single-obfuscation set evaluation

In [26]:
single_df["hybrid_prediction"] = hybrid_predict(
    single_df[single_text_col]
)

In [27]:
single_overall_metrics = calculate_metrics(
    y_single,
    single_df["hybrid_prediction"]
)

single_overall_metrics

{'precision': 1.0,
 'recall': 0.6511965192168238,
 'f1': 0.7887571365832236,
 'accuracy': 0.6511965192168238}

Evaluate by individual attack

In [28]:
single_attack_results = []

if single_attack_col is not None:

    for attack in single_df[single_attack_col].dropna().unique():

        subset = single_df[
            single_df[single_attack_col] == attack
        ]

        metrics = calculate_metrics(
            convert_labels(subset[single_label_col]),
            subset["hybrid_prediction"]
        )

        single_attack_results.append({
            "model": f"Hybrid-{best_ml_name}",
            "attack_type": attack,
            **metrics
        })

single_attack_results = pd.DataFrame(
    single_attack_results
)

single_attack_results

,model,attack_type,precision,recall,f1,accuracy
0,Hybrid-SVM,clean,1.0,0.813441,0.897124,0.813441
1,Hybrid-SVM,character_substitution,1.0,0.771698,0.871139,0.771698
2,Hybrid-SVM,spacing,1.0,0.620095,0.765505,0.620095
3,Hybrid-SVM,symbolic_masking,1.0,0.283266,0.441477,0.283266
4,Hybrid-SVM,misspelling,1.0,0.730684,0.844388,0.730684
5,Hybrid-SVM,unicode_confusable,1.0,0.793243,0.884702,0.793243


Evaluate single attacks by severity

In [29]:
single_severity_results = []

if severity_single_col is not None:

    for severity in single_df[
        severity_single_col
    ].dropna().unique():

        subset = single_df[
            single_df[severity_single_col] == severity
        ]

        metrics = calculate_metrics(
            convert_labels(subset[single_label_col]),
            subset["hybrid_prediction"]
        )

        single_severity_results.append({
            "model": f"Hybrid-{best_ml_name}",
            "dataset": "single",
            "severity": severity,
            **metrics
        })

single_severity_results = pd.DataFrame(
    single_severity_results
)

single_severity_results

,model,dataset,severity,precision,recall,f1,accuracy
0,Hybrid-SVM,single,none,1.0,0.813441,0.897124,0.813441
1,Hybrid-SVM,single,low,1.0,0.744081,0.853264,0.744081
2,Hybrid-SVM,single,medium,1.0,0.635329,0.777005,0.635329
3,Hybrid-SVM,single,high,1.0,0.538922,0.700389,0.538922


Attack × severity analysis

In [30]:
single_detailed_results = []

if (
    single_attack_col is not None and
    severity_single_col is not None
):

    groups = single_df.groupby([
        single_attack_col,
        severity_single_col
    ])

    for (attack, severity), subset in groups:

        metrics = calculate_metrics(
            convert_labels(subset[single_label_col]),
            subset["hybrid_prediction"]
        )

        single_detailed_results.append({
            "model": f"Hybrid-{best_ml_name}",
            "attack_type": attack,
            "severity": severity,
            "n": len(subset),
            **metrics
        })

single_detailed_results = pd.DataFrame(
    single_detailed_results
)

single_detailed_results.head(20)

,model,attack_type,severity,n,precision,recall,f1,accuracy
0,Hybrid-SVM,character_substitution,high,2723,1.0,0.741094,0.851297,0.741094
1,Hybrid-SVM,character_substitution,low,2723,1.0,0.799853,0.888798,0.799853
2,Hybrid-SVM,character_substitution,medium,2723,1.0,0.774146,0.872697,0.774146
3,Hybrid-SVM,clean,none,2723,1.0,0.813441,0.897124,0.813441
4,Hybrid-SVM,misspelling,high,2717,1.0,0.659551,0.794855,0.659551
5,Hybrid-SVM,misspelling,low,2719,1.0,0.789996,0.882679,0.789996
6,Hybrid-SVM,misspelling,medium,2718,1.0,0.742458,0.852196,0.742458
7,Hybrid-SVM,spacing,high,2307,1.0,0.487646,0.655594,0.487646
8,Hybrid-SVM,spacing,low,2585,1.0,0.736557,0.848296,0.736557
9,Hybrid-SVM,spacing,medium,2473,1.0,0.621917,0.766891,0.621917


Multi-obfuscation set evaluation

In [31]:
multi_df["hybrid_prediction"] = hybrid_predict(
    multi_df[multi_text_col]
)

In [32]:
multi_overall_metrics = calculate_metrics(
    y_multi,
    multi_df["hybrid_prediction"]
)

multi_overall_metrics

{'precision': 1.0,
 'recall': 0.4950457317073171,
 'f1': 0.662248279378027,
 'accuracy': 0.4950457317073171}

In [33]:
# Multi severity
multi_severity_results = []

if severity_multi_col is not None:

    for severity in multi_df[
        severity_multi_col
    ].dropna().unique():

        subset = multi_df[
            multi_df[severity_multi_col] == severity
        ]

        metrics = calculate_metrics(
            convert_labels(subset[multi_label_col]),
            subset["hybrid_prediction"]
        )

        multi_severity_results.append({
            "model": f"Hybrid-{best_ml_name}",
            "dataset": "multi",
            "severity": severity,
            **metrics
        })

multi_severity_results = pd.DataFrame(
    multi_severity_results
)

multi_severity_results

,model,dataset,severity,precision,recall,f1,accuracy
0,Hybrid-SVM,multi,low,1.0,0.666916,0.800180,0.666916
1,Hybrid-SVM,multi,medium,1.0,0.482496,0.650924,0.482496
2,Hybrid-SVM,multi,high,1.0,0.329316,0.495467,0.329316


Robustness drop

In [34]:
clean_f1 = clean_hybrid_metrics["f1"]

single_f1 = single_overall_metrics["f1"]
multi_f1 = multi_overall_metrics["f1"]

single_robustness_drop = clean_f1 - single_f1
multi_robustness_drop = clean_f1 - multi_f1

print(
    "Single-obfuscation F1 drop:",
    single_robustness_drop
)

print(
    "Multi-obfuscation F1 drop:",
    multi_robustness_drop
)

Single-obfuscation F1 drop: -0.22095400917999364
Multi-obfuscation F1 drop: -0.09444515197479708


In [35]:
# Relative drop
def relative_drop(clean, attack):

    if clean == 0:
        return np.nan

    return ((clean - attack) / clean) * 100

In [36]:
print(
    "Single relative drop:",
    relative_drop(clean_f1, single_f1)
)

print(
    "Multi relative drop:",
    relative_drop(clean_f1, multi_f1)
)

Single relative drop: -38.91384152646299
Multi relative drop: -16.633432860211443


In [37]:
# Recovery function
def recovery_rate(
    clean_ml_f1,
    attack_ml_f1,
    hybrid_attack_f1
):

    lost = clean_ml_f1 - attack_ml_f1

    recovered = hybrid_attack_f1 - attack_ml_f1

    if lost <= 0:
        return np.nan

    return (recovered / lost) * 100

Evaluation of ML only

In [38]:
if best_ml_name == "SVM":
    ml_model = svm_raw
else:
    ml_model = lr_raw

In [39]:
def ml_only_predict(texts):

    features = raw_vectorizer.transform(
        pd.Series(texts)
        .fillna("")
        .astype(str)
    )

    return ml_model.predict(features)

In [40]:
ml_clean_pred = ml_only_predict(
    test_df[test_text_col]
)

ml_single_pred = ml_only_predict(
    single_df[single_text_col]
)

ml_multi_pred = ml_only_predict(
    multi_df[multi_text_col]
)

In [41]:
ml_clean_metrics = calculate_metrics(
    y_test,
    ml_clean_pred
)

ml_single_metrics = calculate_metrics(
    y_single,
    ml_single_pred
)

ml_multi_metrics = calculate_metrics(
    y_multi,
    ml_multi_pred
)

Hybrid recovery

In [42]:
# Single
single_f1_gain = (
    single_overall_metrics["f1"]
    -
    ml_single_metrics["f1"]
)

single_recovery = recovery_rate(
    ml_clean_metrics["f1"],
    ml_single_metrics["f1"],
    single_overall_metrics["f1"]
)

In [43]:
# Multi
multi_f1_gain = (
    multi_overall_metrics["f1"]
    -
    ml_multi_metrics["f1"]
)

multi_recovery = recovery_rate(
    ml_clean_metrics["f1"],
    ml_multi_metrics["f1"],
    multi_overall_metrics["f1"]
)

In [44]:
print("SINGLE OBFUSCATION")
print("ML F1:", ml_single_metrics["f1"])
print("Hybrid F1:", single_overall_metrics["f1"])
print("F1 recovered:", single_f1_gain)
print("Recovery rate:", single_recovery)

print("\nMULTI OBFUSCATION")
print("ML F1:", ml_multi_metrics["f1"])
print("Hybrid F1:", multi_overall_metrics["f1"])
print("F1 recovered:", multi_f1_gain)
print("Recovery rate:", multi_recovery)

SINGLE OBFUSCATION
ML F1: 0.7738178897701034
Hybrid F1: 0.7887571365832236
F1 recovered: 0.0149392468131202
Recovery rate: nan

MULTI OBFUSCATION
ML F1: 0.6459103810097188
Hybrid F1: 0.662248279378027
F1 recovered: 0.016337898368308235
Recovery rate: nan


Results

In [45]:
hybrid_overall_results = pd.DataFrame([
    {
        "model": f"Hybrid-{best_ml_name}",
        "condition": "clean",
        **clean_hybrid_metrics,
        "robustness_drop": 0,
        "relative_drop_pct": 0
    },

    {
        "model": f"Hybrid-{best_ml_name}",
        "condition": "single_obfuscation",
        **single_overall_metrics,
        "robustness_drop":
            clean_f1 - single_overall_metrics["f1"],
        "relative_drop_pct":
            relative_drop(
                clean_f1,
                single_overall_metrics["f1"]
            )
    },

    {
        "model": f"Hybrid-{best_ml_name}",
        "condition": "multi_obfuscation",
        **multi_overall_metrics,
        "robustness_drop":
            clean_f1 - multi_overall_metrics["f1"],
        "relative_drop_pct":
            relative_drop(
                clean_f1,
                multi_overall_metrics["f1"]
            )
    }
])

hybrid_overall_results

,model,condition,precision,recall,f1,accuracy,robustness_drop,relative_drop_pct
0,Hybrid-SVM,clean,0.436109,0.813441,0.567803,0.851956,0.000000,0.000000
1,Hybrid-SVM,single_obfuscation,1.000000,0.651197,0.788757,0.651197,-0.220954,-38.913842
2,Hybrid-SVM,multi_obfuscation,1.000000,0.495046,0.662248,0.495046,-0.094445,-16.633433


In [46]:
single_errors = single_df.copy()

single_errors["true_label"] = y_single.values

single_errors["correct"] = (
    single_errors["hybrid_prediction"]
    ==
    single_errors["true_label"]
)

single_errors["error_type"] = np.where(
    single_errors["correct"],
    "correct",
    np.where(
        (single_errors["true_label"] == 1) &
        (single_errors["hybrid_prediction"] == 0),
        "false_negative",
        "false_positive"
    )
)

In [47]:
# Failures only
hybrid_errors = single_errors[
    single_errors["correct"] == False
].copy()

hybrid_errors.head()

,id,text,text_length,source_dataset,abusive,original_text,obfuscated_text,obfuscation_type,severity,obfuscated_length,normalised_text,hybrid_prediction,true_label,correct,error_type
6,jigsaw_88344,Just stating the truth. I know how you people ...,245,JIGSAW,OFF,Just stating the truth. I know how you people ...,Ju*t --*__ng t-e tru.h. I k#o. h#w ..u p*op*e ...,symbolic_masking,medium,245,jut --*__ng te truh. i k#o. h#w .u pope msubat...,0,1,False,false_negative
7,jigsaw_88344,Just stating the truth. I know how you people ...,245,JIGSAW,OFF,Just stating the truth. I know how you people ...,.*.. s*a*-n_ *.- trut-. _ .-ow _._ y.. #eop#* ...,symbolic_masking,high,245,.*. san_ *.- trut-. _ .-ow _._ y. #eop#* _.st-...,0,1,False,false_negative
14,hatexplain_9882,maga something has to be done about that scumb...,208,HateXplain,OFF,maga something has to be done about that scumb...,maga something has to be done about that scumb...,clean,none,208,maga something has to be done about that scumb...,0,1,False,false_negative
15,hatexplain_9882,maga something has to be done about that scumb...,208,HateXplain,OFF,maga something has to be done about that scumb...,mag@ somethin9 has to be don3 @bout that scumb...,character_substitution,low,208,maga somethin9 has to be done about that scumb...,0,1,False,false_negative
16,hatexplain_9882,maga something has to be done about that scumb...,208,HateXplain,OFF,maga something has to be done about that scumb...,maga $0me7hing ha$ 70 be don3 ab0u7 that scum8...,character_substitution,medium,208,maga something has to be done about that scum8...,0,1,False,false_negative


In [48]:
hybrid_errors.to_csv(
    "hybrid_error_analysis.csv",
    index=False
)

In [49]:
hybrid_overall_results.to_csv(
    "hybrid_overall_results.csv",
    index=False
)

single_attack_results.to_csv(
    "hybrid_single_obfuscation_results.csv",
    index=False
)

single_severity_results.to_csv(
    "hybrid_single_severity_results.csv",
    index=False
)

single_detailed_results.to_csv(
    "hybrid_single_detailed_results.csv",
    index=False
)

multi_severity_results.to_csv(
    "hybrid_multi_severity_results.csv",
    index=False
)

single_df.to_csv(
    "hybrid_single_predictions.csv",
    index=False
)

multi_df.to_csv(
    "hybrid_multi_predictions.csv",
    index=False
)

In [50]:
joblib.dump(
    hybrid_vectorizer,
    "hybrid_tfidf_vectorizer.joblib"
)

joblib.dump(
    hybrid_model,
    "hybrid_model.joblib"
)

['hybrid_model.joblib']

In [51]:
import shutil

files_to_zip = [
    "hybrid_overall_results.csv",
    "hybrid_single_obfuscation_results.csv",
    "hybrid_single_severity_results.csv",
    "hybrid_single_detailed_results.csv",
    "hybrid_multi_severity_results.csv",
    "hybrid_single_predictions.csv",
    "hybrid_multi_predictions.csv",
    "hybrid_error_analysis.csv",
    "hybrid_tfidf_vectorizer.joblib",
    "hybrid_model.joblib"
]

os.makedirs("hybrid_results", exist_ok=True)

for f in files_to_zip:
    if os.path.exists(f):
        shutil.copy(f, "hybrid_results")

shutil.make_archive(
    "hybrid_results",
    "zip",
    "hybrid_results"
)

from google.colab import files
files.download("hybrid_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>